In [1]:
# 1. Librerías
!pip install -q -U transformers accelerate

# 2. Modelo
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")
print("GPU:", torch.cuda.get_device_name(0))

# 3. Datos GSM8K (descarga directa, sin la librería datasets)
import requests, json
url = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"
datos = [json.loads(l) for l in requests.get(url).text.strip().split("\n")]
print("Problemas:", len(datos))

# 4. Funciones de corrección
import re
def extraer_numero(texto):
    m = re.findall(r"\\boxed\{([^}]*)\}", texto)
    cand = m[-1] if m else (re.findall(r"-?\d[\d,]*\.?\d*", texto) or [""])[-1]
    return cand.replace(",", "").replace("$", "").strip()
def numero_gold(a):
    return a.split("####")[-1].replace(",", "").replace("$", "").strip()
def igual(a, b):
    try: return abs(float(a) - float(b)) < 1e-6
    except: return a.strip() == b.strip()

# 5. Medición (guarda cada respuesta; reanudable)
import os
from tqdm import tqdm
ARCHIVO = "gsm8k_base.jsonl"
hechos = 0
if os.path.exists(ARCHIVO):
    with open(ARCHIVO) as f:
        for l in f:
            try: json.loads(l); hechos += 1
            except: break
N = 1000
with open(ARCHIVO, "a", encoding="utf-8") as f:
    for i in tqdm(range(hechos, N)):
        ej = datos[i]
        entrada = tokenizer.apply_chat_template(
            [{"role": "user", "content": ej["question"]}],
            add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        salida = model.generate(**entrada, max_new_tokens=1024, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
        texto = tokenizer.decode(salida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)
        pred, gold = extraer_numero(texto), numero_gold(ej["answer"])
        f.write(json.dumps({"i": i, "pregunta": ej["question"], "respuesta": texto,
                            "prediccion": pred, "correcta": gold, "acierto": igual(pred, gold)},
                           ensure_ascii=False) + "\n")
        f.flush()

# 6. Número + documento completo
import datetime
registros = [json.loads(l) for l in open(ARCHIVO, encoding="utf-8")]
aciertos, total = sum(r["acierto"] for r in registros), len(registros)
print(f"PRECISIÓN BASE: {aciertos}/{total} = {aciertos/total*100:.2f}%")
with open("baseline_gsm8k_completo.md", "w", encoding="utf-8") as f:
    f.write(f"# Baseline GSM8K — DeepSeek-R1-Distill-Qwen-1.5B (base)\n\n")
    f.write(f"- Fecha: {datetime.date.today()} · N: {total} · greedy · max_new_tokens=1024\n")
    f.write(f"- **Precisión: {aciertos}/{total} = {aciertos/total*100:.2f}%**\n\n---\n\n")
    for r in registros:
        estado = "CORRECTO" if r["acierto"] else "INCORRECTO"
        f.write(f"## Problema {r['i']} — {estado}\n\n**Pregunta:**\n\n{r['pregunta']}\n\n")
        f.write(f"**Respuesta del modelo:**\n\n{r['respuesta']}\n\n")
        f.write(f"**Predicción:** `{r['prediccion']}` · **Correcta:** `{r['correcta']}`\n\n---\n\n")
print("Listo:", aciertos, "/", total)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

GPU: Tesla T4
Problemas: 1319


100%|██████████| 1000/1000 [5:53:52<00:00, 21.23s/it]

PRECISIÓN BASE: 570/1000 = 57.00%
Listo: 570 / 1000
